In [ ]:
import pandas as pd
import textwrap
from experiment_runner import LLMExperimentRunner, Experiment

Persona generation (sample of 10)

In [2]:
df = pd.read_csv("Synthetic_Big5_data.csv")
df = df.round(2)
df = df.sample(n=10, random_state=1)
df

,Enthusiasm,Assertiveness,Compassion,Politeness,Industriousness,Orderliness,Volatility,Withdrawal,Intellect,Openness,gender,Extraversion,Agreeableness,Conscientiousness,Neuroticism,Openness_Intellect
507,3.89,2.96,4.24,4.10,3.21,3.64,1.92,3.22,1.91,2.95,female,3.42,4.17,3.43,2.57,2.43
818,5.28,4.09,4.77,4.33,2.65,3.53,2.23,2.90,3.10,3.10,female,4.68,4.55,3.09,2.57,3.10
452,3.37,3.15,4.27,4.44,2.53,3.37,2.17,2.93,3.01,3.00,male,3.26,4.35,2.95,2.55,3.01
368,2.75,3.43,3.64,3.56,2.70,2.94,2.66,2.90,4.28,4.52,male,3.09,3.60,2.82,2.78,4.40
242,5.05,4.06,3.85,3.25,2.98,3.26,2.00,1.83,3.72,3.73,male,4.56,3.55,3.12,1.91,3.72
929,3.85,2.96,3.43,2.56,2.80,2.92,3.62,3.03,2.66,2.96,female,3.41,2.99,2.86,3.32,2.81
262,2.96,3.74,3.68,3.32,2.35,3.21,2.39,3.07,3.35,3.81,male,3.35,3.50,2.78,2.73,3.58
810,3.53,3.04,4.60,3.75,2.69,3.13,3.08,3.71,3.18,4.49,female,3.29,4.18,2.91,3.39,3.84
318,3.02,3.01,4.00,3.59,3.11,3.30,2.76,2.94,2.81,2.96,male,3.02,3.79,3.20,2.85,2.88
49,4.64,3.09,3.91,2.94,3.23,2.96,2.93,2.72,3.84,3.14,male,3.87,3.43,3.10,2.82,3.49


In [11]:
PERSONA_SYSTEM_PROMPT = (
"""You are a professional personality analyst and storyteller. Your task is to generate a **realistic, detailed personality profile** of a single individual based on their **Big Five subfacet scores on a 1-5 Likert scale**.

The subfacets are:

- Extraversion: Enthusiasm, Assertiveness
- Agreeableness: Compassion, Politeness
- Conscientiousness: Industriousness, Orderliness
- Neuroticism: Volatility, Withdrawal
- Openness to Experience: Intellect, Openness

Instructions:

1. Input will provide numeric scores (1-5) for each subfacet.  

2. Output should be a **detailed personality description (~800 words)** that represents a **coherent, realistic individual** consistent with the scores.  

3. Use the subfacet scores to shape the description:
- High scores (4-5) → emphasize frequent, strong expression of the trait  
- Medium scores (~3) → portray moderate, balanced expression  
- Low scores (1-2) → portray subtle or infrequent expression  

4. Include, organically in the narrative:
- Behavior patterns in social, work, and personal contexts  
- Typical emotional reactions and coping styles  
- Interaction style with others, including assertiveness, empathy, and politeness  
- Cognitive tendencies, interests, and openness to new experiences  
- Habits, routines, and level of conscientiousness  

5. The output should read like a **psychologically coherent personality profile**, not just a list of traits. Do not include raw scores in the text.

6. The text should be suitable to be used as a **persona description** for an LLM that will later simulate the behavior of this individual.

7. The tone should be **formal but narrative**, suitable for psychology research, character modeling, or behavioral simulation.

8. Always produce a **single continuous text**, approximately 800 words.

9. Do NOT include a Title or description of what you are about to do, just immediatly start with the Persona description

10. Do NOT include a summary at the end

In the next message you will recieve the personality trait messages:"""
) 

In [ ]:
runner = LLMExperimentRunner(llm_model="deepseek-chat V3.2", max_workers=10)
df_persona = runner.generate_personas(df, PERSONA_SYSTEM_PROMPT)

In [ ]:
df_persona

Experiment Setup:

In [ ]:
# maybe do sth like this to remove the color as a confounder
# import random
# red_black = random.uniform(0, 1)
# if red_black < 0.5:
#     print("red")
# else:
#     print("black")

In [5]:
# defined in experiment_runner
# @dataclass(frozen=True) # make it immutable
# class Experiment:
#     prompt: str
#     sys_instruction: str


def gamblers_fallacy_experiments(max_previous_outcomes: int):
    BASE_PROMPT = (
    "The person you are imitating is playing roulette and can either bet on black or red."
    "{history} What would the person choose?"
    )
    GAMBLERS_FALLACY_SYS_INSTRUCTION = "Provide a single word answer with either 'black' or 'red' (without the quotes)." # THIS could also be part of the regular prompt?
    experiments = []

    for n in range(0, max_previous_outcomes + 1):
        if n == 0:
            history = ""
        elif n == 1:
            history = "The last outcome was red."
        else:
            history = f"The last {n} outcomes were red."

        experiments.append({
            f"Gamblers_Fallacy_{n}": Experiment(
                prompt=BASE_PROMPT.format(history=history),
                sys_instruction=GAMBLERS_FALLACY_SYS_INSTRUCTION
            )
        })

    return experiments

def clean_prompt(prompt: str) -> str:
    p = textwrap.dedent(prompt).strip()
    p = " ".join(line.strip() for line in p.splitlines())
    return p

LOSS_AVERSION_SYS_INSTRUCTION = "You take part in a survey. You can answer with either 'A' or 'B' (without the quotes)." # maybe adjust

experiment_instructions = gamblers_fallacy_experiments(max_previous_outcomes=4) # add back later -> + [
#     {
#         "Loss_Aversion_1": Experiment(
#             prompt=(
#                 "A: (2,500 with probability .33, 2,400 with probability .66, 0 with probability .01)\n "
#                 "B: (2,400 with certainty)"
#                 ),
#             sys_instruction=LOSS_AVERSION_SYS_INSTRUCTION
#         )
#     },
#     {
#         "Loss_Aversion_2": Experiment(
#             prompt=(
#                 "A: (2,500 with probability .33, 0 with probability .67)\n "
#                 "B: (2,400 with probability .34, 0 with probability .66)"
#                 ),
#             sys_instruction=LOSS_AVERSION_SYS_INSTRUCTION
#         )
#     },
#     {
#         "Loss_Aversion_3": Experiment(
#             prompt=(
#                 "A: (4,000 with probability .80)\n "
#                 "B: (3,000 with certainty)"
#                 ),
#             sys_instruction=LOSS_AVERSION_SYS_INSTRUCTION
#         )
#     },
#     {
#         "Loss_Aversion_4": Experiment(
#             prompt=(
#                 "A: (4,000 with probability .20)\n "
#                 "B: (3,000 with probability .25)"
#                 ),
#             sys_instruction=LOSS_AVERSION_SYS_INSTRUCTION
#         )
#     },{
#         "Loss_Aversion_5": Experiment(
#             prompt=(
#                 "A: (50% chance to win a three-week tour of England, France, and Italy)\n "
#                 "B: (A one-week tour of England, with certainty)"
#                 ),
#             sys_instruction=LOSS_AVERSION_SYS_INSTRUCTION
#         )
#     },
#     {
#         "Loss_Aversion_6": Experiment(
#             prompt=(
#                 "A: (5% chance to win a three-week tour of England, France, and Italy)\n "
#                 "B: (10% chance to win a one-week tour of England)"
#                 ),
#             sys_instruction=LOSS_AVERSION_SYS_INSTRUCTION
#         )
#     },{
#         "Loss_Aversion_7": Experiment(
#             prompt=(
#                 "A: (6,000 with probability .45)\n "
#                 "B: (3,000 with probability .90)"
#                 ),
#             sys_instruction=LOSS_AVERSION_SYS_INSTRUCTION
#         )
#     },
#     {
#         "Loss_Aversion_8": Experiment(
#             prompt=(
#                 "A: (6,000 with probability .001)\n "
#                 "B: (3,000 with probability .002)"
#                 ),
#             sys_instruction=LOSS_AVERSION_SYS_INSTRUCTION
#         )
#     },
#     {
#         "Loss_Aversion_9": Experiment(
#             prompt=clean_prompt(
#                 """
#                 Suppose you consider the possibility of insuring some property
#                 against damage, e.g., fire or theft. After examining the risks and the premium you
#                 find that you have no clear preference between the options of purchasing
#                 insurance or leaving the property uninsured.
#                 It is then called to your attention that the insurance company offers a new
#                 program called probabilistic insurance. In this program you pay half of the regular
#                 premium. In case of damage, there is a 50 per cent chance that you pay the other
#                 half of the premium and the insurance company covers all the losses; and there is a
#                 50 per cent chance that you get back your insurance payment and suffer all the
#                 losses. For example, if an accident occurs on an odd day of the month, you pay the
#                 other half of the regular premium and your losses are covered; but if the accident
#                 occurs on an even day of the month, your insurance payment is refunded and your
#                 losses are not covered.
#                 Recall that the premium for full coverage is such that you find this insurance
#                 barely worth its cost.
#                 Under these circumstances, would you purchase probabilistic insurance:
#                 A: Yes
#                 B: No
#                 """
#                 ),
#             sys_instruction=LOSS_AVERSION_SYS_INSTRUCTION
#         )
#     },
#     {# CONTINUE WORKING HERE:
#         "Loss_Aversion_10": Experiment(
#             prompt=(
#                 "A: (6,000 with probability .001)\n "
#                 "B: (3,000 with probability .002)"
#                 ),
#             sys_instruction=LOSS_AVERSION_SYS_INSTRUCTION
#         )
#     }
# ]

In [ ]:
# "The person you are imitating is playing roulette and can either bet on red or black."
# "{history} What would the person choose?"
df_persona = pd.read_csv("persona_checkpoint.csv")
runner = LLMExperimentRunner(llm_model="deepseek-chat V3.2", max_workers=10, temperature=1.0)
df_experiment = runner.run_experiment(df_persona, instructions=experiment_instructions, use_persona=True)
df_experiment

Experiments: 100%|██████████| 5/5 [00:14<00:00,  2.84s/it]


,Enthusiasm,Assertiveness,Compassion,Politeness,Industriousness,Orderliness,Volatility,Withdrawal,Intellect,Openness,...,Gamblers_Fallacy_0_experiment_deepseek-chat V3.2,Gamblers_Fallacy_0_experiment_deepseek-chat V3.2_date,Gamblers_Fallacy_1_experiment_deepseek-chat V3.2,Gamblers_Fallacy_1_experiment_deepseek-chat V3.2_date,Gamblers_Fallacy_2_experiment_deepseek-chat V3.2,Gamblers_Fallacy_2_experiment_deepseek-chat V3.2_date,Gamblers_Fallacy_3_experiment_deepseek-chat V3.2,Gamblers_Fallacy_3_experiment_deepseek-chat V3.2_date,Gamblers_Fallacy_4_experiment_deepseek-chat V3.2,Gamblers_Fallacy_4_experiment_deepseek-chat V3.2_date
0,3.89,2.96,4.24,4.10,3.21,3.64,1.92,3.22,1.91,2.95,...,black,2026-02-03 17:37:41+01:00,black,2026-02-03 17:37:42+01:00,black,2026-02-03 17:37:44+01:00,black,2026-02-03 17:37:46+01:00,black,2026-02-03 17:37:48+01:00
1,5.28,4.09,4.77,4.33,2.65,3.53,2.23,2.90,3.10,3.10,...,black,2026-02-03 17:37:41+01:00,black,2026-02-03 17:37:42+01:00,black,2026-02-03 17:37:44+01:00,black,2026-02-03 17:37:46+01:00,black,2026-02-03 17:37:48+01:00
2,3.37,3.15,4.27,4.44,2.53,3.37,2.17,2.93,3.01,3.00,...,black,2026-02-03 17:37:41+01:00,black,2026-02-03 17:37:42+01:00,black,2026-02-03 17:37:44+01:00,black,2026-02-03 17:37:46+01:00,black,2026-02-03 17:37:48+01:00
3,2.75,3.43,3.64,3.56,2.70,2.94,2.66,2.90,4.28,4.52,...,black,2026-02-03 17:37:41+01:00,black,2026-02-03 17:37:42+01:00,black,2026-02-03 17:37:44+01:00,black,2026-02-03 17:37:46+01:00,black,2026-02-03 17:37:48+01:00
4,5.05,4.06,3.85,3.25,2.98,3.26,2.00,1.83,3.72,3.73,...,red,2026-02-03 17:37:41+01:00,black,2026-02-03 17:37:42+01:00,black,2026-02-03 17:37:44+01:00,black,2026-02-03 17:37:46+01:00,black,2026-02-03 17:37:48+01:00
5,3.85,2.96,3.43,2.56,2.80,2.92,3.62,3.03,2.66,2.96,...,black,2026-02-03 17:37:41+01:00,black,2026-02-03 17:37:42+01:00,black,2026-02-03 17:37:44+01:00,black,2026-02-03 17:37:46+01:00,black,2026-02-03 17:37:48+01:00
6,2.96,3.74,3.68,3.32,2.35,3.21,2.39,3.07,3.35,3.81,...,black,2026-02-03 17:37:41+01:00,black,2026-02-03 17:37:42+01:00,black,2026-02-03 17:37:44+01:00,black,2026-02-03 17:37:46+01:00,black,2026-02-03 17:37:48+01:00
7,3.53,3.04,4.60,3.75,2.69,3.13,3.08,3.71,3.18,4.49,...,black,2026-02-03 17:37:41+01:00,black,2026-02-03 17:37:42+01:00,black,2026-02-03 17:37:44+01:00,black,2026-02-03 17:37:46+01:00,black,2026-02-03 17:37:48+01:00
8,3.02,3.01,4.00,3.59,3.11,3.30,2.76,2.94,2.81,2.96,...,black,2026-02-03 17:37:41+01:00,black,2026-02-03 17:37:42+01:00,black,2026-02-03 17:37:44+01:00,black,2026-02-03 17:37:46+01:00,black,2026-02-03 17:37:48+01:00
9,4.64,3.09,3.91,2.94,3.23,2.96,2.93,2.72,3.84,3.14,...,red,2026-02-03 17:37:41+01:00,black,2026-02-03 17:37:42+01:00,black,2026-02-03 17:37:44+01:00,black,2026-02-03 17:37:46+01:00,black,2026-02-03 17:37:48+01:00


In [ ]:
# "The person you are imitating is playing roulette and can either bet on black or red." (for the first bet it almost always bets on the last color mentioned in the prompt)
# "{history} What would the person choose?"
df_persona = pd.read_csv("persona_checkpoint.csv")
runner = LLMExperimentRunner(llm_model="deepseek-chat V3.2", max_workers=10, temperature=1.0)
df_experiment = runner.run_experiment(df_persona, instructions=experiment_instructions, use_persona=True, save_name="red_last")
df_experiment

Experiments: 100%|██████████| 5/5 [00:08<00:00,  1.67s/it]


,Enthusiasm,Assertiveness,Compassion,Politeness,Industriousness,Orderliness,Volatility,Withdrawal,Intellect,Openness,...,Gamblers_Fallacy_0_experiment_deepseek-chat V3.2,Gamblers_Fallacy_0_experiment_deepseek-chat V3.2_date,Gamblers_Fallacy_1_experiment_deepseek-chat V3.2,Gamblers_Fallacy_1_experiment_deepseek-chat V3.2_date,Gamblers_Fallacy_2_experiment_deepseek-chat V3.2,Gamblers_Fallacy_2_experiment_deepseek-chat V3.2_date,Gamblers_Fallacy_3_experiment_deepseek-chat V3.2,Gamblers_Fallacy_3_experiment_deepseek-chat V3.2_date,Gamblers_Fallacy_4_experiment_deepseek-chat V3.2,Gamblers_Fallacy_4_experiment_deepseek-chat V3.2_date
0,3.89,2.96,4.24,4.10,3.21,3.64,1.92,3.22,1.91,2.95,...,red,2026-02-03 18:07:05+01:00,black,2026-02-03 18:07:07+01:00,red,2026-02-03 18:07:08+01:00,black,2026-02-03 18:07:10+01:00,red,2026-02-03 18:07:11+01:00
1,5.28,4.09,4.77,4.33,2.65,3.53,2.23,2.90,3.10,3.10,...,red,2026-02-03 18:07:05+01:00,red,2026-02-03 18:07:07+01:00,red,2026-02-03 18:07:08+01:00,red,2026-02-03 18:07:10+01:00,red,2026-02-03 18:07:11+01:00
2,3.37,3.15,4.27,4.44,2.53,3.37,2.17,2.93,3.01,3.00,...,red,2026-02-03 18:07:05+01:00,black,2026-02-03 18:07:07+01:00,red,2026-02-03 18:07:09+01:00,black,2026-02-03 18:07:10+01:00,red,2026-02-03 18:07:11+01:00
3,2.75,3.43,3.64,3.56,2.70,2.94,2.66,2.90,4.28,4.52,...,red,2026-02-03 18:07:05+01:00,black,2026-02-03 18:07:07+01:00,black,2026-02-03 18:07:09+01:00,black,2026-02-03 18:07:10+01:00,black,2026-02-03 18:07:11+01:00
4,5.05,4.06,3.85,3.25,2.98,3.26,2.00,1.83,3.72,3.73,...,red,2026-02-03 18:07:05+01:00,black,2026-02-03 18:07:07+01:00,black,2026-02-03 18:07:09+01:00,red,2026-02-03 18:07:10+01:00,red,2026-02-03 18:07:11+01:00
5,3.85,2.96,3.43,2.56,2.80,2.92,3.62,3.03,2.66,2.96,...,red,2026-02-03 18:07:05+01:00,black,2026-02-03 18:07:07+01:00,black,2026-02-03 18:07:09+01:00,red,2026-02-03 18:07:10+01:00,red,2026-02-03 18:07:11+01:00
6,2.96,3.74,3.68,3.32,2.35,3.21,2.39,3.07,3.35,3.81,...,red,2026-02-03 18:07:05+01:00,black,2026-02-03 18:07:07+01:00,red,2026-02-03 18:07:09+01:00,black,2026-02-03 18:07:10+01:00,red,2026-02-03 18:07:11+01:00
7,3.53,3.04,4.60,3.75,2.69,3.13,3.08,3.71,3.18,4.49,...,red,2026-02-03 18:07:05+01:00,black,2026-02-03 18:07:07+01:00,black,2026-02-03 18:07:09+01:00,red,2026-02-03 18:07:10+01:00,black,2026-02-03 18:07:11+01:00
8,3.02,3.01,4.00,3.59,3.11,3.30,2.76,2.94,2.81,2.96,...,red,2026-02-03 18:07:05+01:00,black,2026-02-03 18:07:07+01:00,black,2026-02-03 18:07:09+01:00,black,2026-02-03 18:07:10+01:00,black,2026-02-03 18:07:11+01:00
9,4.64,3.09,3.91,2.94,3.23,2.96,2.93,2.72,3.84,3.14,...,red,2026-02-03 18:07:05+01:00,black,2026-02-03 18:07:07+01:00,red,2026-02-03 18:07:09+01:00,red,2026-02-03 18:07:10+01:00,red,2026-02-03 18:07:11+01:00


In [ ]:
# no Persona
df_persona = pd.read_csv("persona_checkpoint.csv")
runner = LLMExperimentRunner(llm_model="deepseek-chat V3.2", max_workers=10, temperature=1.0)
df_experiment = runner.run_experiment(df_persona, instructions=experiment_instructions, use_persona=False, save_name="black_last")
df_experiment

Experiments: 100%|██████████| 5/5 [00:07<00:00,  1.46s/it]


,Enthusiasm,Assertiveness,Compassion,Politeness,Industriousness,Orderliness,Volatility,Withdrawal,Intellect,Openness,...,Gamblers_Fallacy_0_experiment_deepseek-chat V3.2,Gamblers_Fallacy_0_experiment_deepseek-chat V3.2_date,Gamblers_Fallacy_1_experiment_deepseek-chat V3.2,Gamblers_Fallacy_1_experiment_deepseek-chat V3.2_date,Gamblers_Fallacy_2_experiment_deepseek-chat V3.2,Gamblers_Fallacy_2_experiment_deepseek-chat V3.2_date,Gamblers_Fallacy_3_experiment_deepseek-chat V3.2,Gamblers_Fallacy_3_experiment_deepseek-chat V3.2_date,Gamblers_Fallacy_4_experiment_deepseek-chat V3.2,Gamblers_Fallacy_4_experiment_deepseek-chat V3.2_date
0,3.89,2.96,4.24,4.10,3.21,3.64,1.92,3.22,1.91,2.95,...,red,2026-02-03 17:39:26+01:00,black,2026-02-03 17:39:28+01:00,black,2026-02-03 17:39:29+01:00,black,2026-02-03 17:39:30+01:00,black,2026-02-03 17:39:32+01:00
1,5.28,4.09,4.77,4.33,2.65,3.53,2.23,2.90,3.10,3.10,...,black,2026-02-03 17:39:26+01:00,black,2026-02-03 17:39:28+01:00,black,2026-02-03 17:39:29+01:00,black,2026-02-03 17:39:30+01:00,black,2026-02-03 17:39:32+01:00
2,3.37,3.15,4.27,4.44,2.53,3.37,2.17,2.93,3.01,3.00,...,black,2026-02-03 17:39:26+01:00,black,2026-02-03 17:39:28+01:00,black,2026-02-03 17:39:29+01:00,black,2026-02-03 17:39:30+01:00,black,2026-02-03 17:39:32+01:00
3,2.75,3.43,3.64,3.56,2.70,2.94,2.66,2.90,4.28,4.52,...,red,2026-02-03 17:39:26+01:00,black,2026-02-03 17:39:28+01:00,black,2026-02-03 17:39:29+01:00,black,2026-02-03 17:39:30+01:00,red,2026-02-03 17:39:32+01:00
4,5.05,4.06,3.85,3.25,2.98,3.26,2.00,1.83,3.72,3.73,...,black,2026-02-03 17:39:26+01:00,black,2026-02-03 17:39:28+01:00,black,2026-02-03 17:39:29+01:00,black,2026-02-03 17:39:30+01:00,black,2026-02-03 17:39:32+01:00
5,3.85,2.96,3.43,2.56,2.80,2.92,3.62,3.03,2.66,2.96,...,black,2026-02-03 17:39:26+01:00,black,2026-02-03 17:39:28+01:00,black,2026-02-03 17:39:29+01:00,black,2026-02-03 17:39:30+01:00,black,2026-02-03 17:39:32+01:00
6,2.96,3.74,3.68,3.32,2.35,3.21,2.39,3.07,3.35,3.81,...,black,2026-02-03 17:39:26+01:00,black,2026-02-03 17:39:28+01:00,black,2026-02-03 17:39:29+01:00,black,2026-02-03 17:39:30+01:00,black,2026-02-03 17:39:32+01:00
7,3.53,3.04,4.60,3.75,2.69,3.13,3.08,3.71,3.18,4.49,...,black,2026-02-03 17:39:26+01:00,black,2026-02-03 17:39:28+01:00,black,2026-02-03 17:39:29+01:00,black,2026-02-03 17:39:30+01:00,black,2026-02-03 17:39:32+01:00
8,3.02,3.01,4.00,3.59,3.11,3.30,2.76,2.94,2.81,2.96,...,black,2026-02-03 17:39:26+01:00,black,2026-02-03 17:39:28+01:00,black,2026-02-03 17:39:29+01:00,black,2026-02-03 17:39:30+01:00,black,2026-02-03 17:39:32+01:00
9,4.64,3.09,3.91,2.94,3.23,2.96,2.93,2.72,3.84,3.14,...,black,2026-02-03 17:39:26+01:00,black,2026-02-03 17:39:28+01:00,black,2026-02-03 17:39:29+01:00,black,2026-02-03 17:39:31+01:00,black,2026-02-03 17:39:32+01:00


In [6]:
df_persona = pd.read_csv("persona_checkpoint.csv")
runner = LLMExperimentRunner(llm_model="deepseek-chat V3.2", max_workers=10, temperature=1.0)
df_experiment = runner.run_experiment(df_persona, instructions=experiment_instructions, use_persona=False, save_name="red_last")
df_experiment

Experiments: 100%|██████████| 5/5 [00:08<00:00,  1.72s/it]


,Enthusiasm,Assertiveness,Compassion,Politeness,Industriousness,Orderliness,Volatility,Withdrawal,Intellect,Openness,...,Gamblers_Fallacy_0_experiment_deepseek-chat V3.2,Gamblers_Fallacy_0_experiment_deepseek-chat V3.2_date,Gamblers_Fallacy_1_experiment_deepseek-chat V3.2,Gamblers_Fallacy_1_experiment_deepseek-chat V3.2_date,Gamblers_Fallacy_2_experiment_deepseek-chat V3.2,Gamblers_Fallacy_2_experiment_deepseek-chat V3.2_date,Gamblers_Fallacy_3_experiment_deepseek-chat V3.2,Gamblers_Fallacy_3_experiment_deepseek-chat V3.2_date,Gamblers_Fallacy_4_experiment_deepseek-chat V3.2,Gamblers_Fallacy_4_experiment_deepseek-chat V3.2_date
0,3.89,2.96,4.24,4.10,3.21,3.64,1.92,3.22,1.91,2.95,...,red,2026-02-03 18:31:03+01:00,black,2026-02-03 18:31:05+01:00,red,2026-02-03 18:31:07+01:00,red,2026-02-03 18:31:08+01:00,black,2026-02-03 18:31:10+01:00
1,5.28,4.09,4.77,4.33,2.65,3.53,2.23,2.90,3.10,3.10,...,red,2026-02-03 18:31:03+01:00,black,2026-02-03 18:31:05+01:00,black,2026-02-03 18:31:07+01:00,red,2026-02-03 18:31:08+01:00,black,2026-02-03 18:31:10+01:00
2,3.37,3.15,4.27,4.44,2.53,3.37,2.17,2.93,3.01,3.00,...,black,2026-02-03 18:31:03+01:00,black,2026-02-03 18:31:05+01:00,black,2026-02-03 18:31:07+01:00,red,2026-02-03 18:31:08+01:00,black,2026-02-03 18:31:10+01:00
3,2.75,3.43,3.64,3.56,2.70,2.94,2.66,2.90,4.28,4.52,...,black,2026-02-03 18:31:03+01:00,red,2026-02-03 18:31:05+01:00,black,2026-02-03 18:31:07+01:00,black,2026-02-03 18:31:08+01:00,black,2026-02-03 18:31:10+01:00
4,5.05,4.06,3.85,3.25,2.98,3.26,2.00,1.83,3.72,3.73,...,black,2026-02-03 18:31:03+01:00,black,2026-02-03 18:31:05+01:00,black,2026-02-03 18:31:07+01:00,black,2026-02-03 18:31:08+01:00,black,2026-02-03 18:31:10+01:00
5,3.85,2.96,3.43,2.56,2.80,2.92,3.62,3.03,2.66,2.96,...,black,2026-02-03 18:31:03+01:00,black,2026-02-03 18:31:05+01:00,black,2026-02-03 18:31:07+01:00,black,2026-02-03 18:31:09+01:00,black,2026-02-03 18:31:10+01:00
6,2.96,3.74,3.68,3.32,2.35,3.21,2.39,3.07,3.35,3.81,...,red,2026-02-03 18:31:03+01:00,black,2026-02-03 18:31:05+01:00,black,2026-02-03 18:31:07+01:00,red,2026-02-03 18:31:08+01:00,red,2026-02-03 18:31:10+01:00
7,3.53,3.04,4.60,3.75,2.69,3.13,3.08,3.71,3.18,4.49,...,black,2026-02-03 18:31:03+01:00,black,2026-02-03 18:31:05+01:00,black,2026-02-03 18:31:07+01:00,black,2026-02-03 18:31:08+01:00,red,2026-02-03 18:31:10+01:00
8,3.02,3.01,4.00,3.59,3.11,3.30,2.76,2.94,2.81,2.96,...,red,2026-02-03 18:31:04+01:00,black,2026-02-03 18:31:05+01:00,red,2026-02-03 18:31:07+01:00,black,2026-02-03 18:31:08+01:00,red,2026-02-03 18:31:10+01:00
9,4.64,3.09,3.91,2.94,3.23,2.96,2.93,2.72,3.84,3.14,...,black,2026-02-03 18:31:03+01:00,black,2026-02-03 18:31:05+01:00,red,2026-02-03 18:31:07+01:00,red,2026-02-03 18:31:08+01:00,red,2026-02-03 18:31:10+01:00
